# EDA: initial inspection (MIT-BIH record 100)

This notebook loads a single WFDB record from the local **MIT-BIH** extract under `data/raw/`, prints basic header fields, and plots the **first 10 seconds** of both ECG channels. Annotation samples from the `atr` file are drawn as vertical guides; in this database they align with beat labels near the QRS complex (commonly used as R-peak timing for beat classification).

## Prerequisite

Run `python -m src.data.download_dataset` after setup (`pip install -r requirements.txt` and `pip install -e .`) so WFDB files exist. The next cell resolves `record_dir` via `src.config.mitdb_record_dir()`.

In [ ]:
from __future__ import annotations
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import wfdb

from src.config import mitdb_record_dir

RECORD_NAME = "100"
record_dir = str(mitdb_record_dir())
record_dir

In [ ]:
record_path = Path(record_dir) / RECORD_NAME

record_path_str = record_path.as_posix()
record = wfdb.rdrecord(record_path_str)
ann = wfdb.rdann(record_path_str, "atr")

## Header metadata

Sampling frequency `fs`, channel count, names, and segment length in samples.

In [ ]:
print("fs:", record.fs)
print("n_sig:", record.n_sig)
print("sig_name:", record.sig_name)
print("sig_len:", record.sig_len)
print("duration_s:", record.sig_len / record.fs)

## Ten-second plot with annotation markers

We slice the first `10 * fs` samples and overlay vertical lines at annotation times that fall inside the window. Symbols follow MIT-BIH beat labels (for example normal `N`, ventricular `V`, and others).

In [ ]:
fs = float(record.fs)
window = min(int(fs * 10), int(record.sig_len))
t = np.arange(window) / fs
sig = record.p_signal[:window, :]

fig, axes = plt.subplots(record.n_sig, 1, figsize=(12, 6), sharex=True)
if record.n_sig == 1:
    axes = [axes]

for i in range(record.n_sig):
    axes[i].plot(t, sig[:, i], color="tab:blue", linewidth=0.9)
    axes[i].set_ylabel(record.sig_name[i])
    axes[i].grid(True, alpha=0.3)

for sample in ann.sample:
    if sample >= window:
        continue
    x = sample / fs
    for ax in axes:
        ax.axvline(x, color="tab:red", alpha=0.35, linewidth=0.9)

axes[-1].set_xlabel("Time (s)")
fig.suptitle(f"MIT-BIH {RECORD_NAME}: first 10 s (two channels) with atr sample times")
fig.tight_layout()
plt.show()